# Entrenamiento temporal — J / Z / other

Video completo entra como secuencia; MediaPipe localiza la mano, se genera un recorte fijo del clip, se calcula flujo óptico Farneback y se clasifica con CNN 3D. El dataset debe estar en `proyecto-ia2/data/videos_jz/{J,Z,other}/<signer_id>/<clip>`. Se conserva el signante para separar train/validation/test por persona.

**Reanudación:** extracción de flujo incremental con manifiesto en Drive; `BackupAndRestore`, checkpoint por época y mejor modelo. Si se interrumpe, vuelve a ejecutar configuración, preprocesamiento y entrenamiento. No borres `backup/`, `checkpoints/` ni `split_balanced_v2.csv`. Este notebook busca un reparto por signante cercano a 70/15/15 para cada clase y exige al menos cinco clips por clase en validation y test. La carpeta de exportación v2 evita reutilizar por accidente el modelo anterior.

In [ ]:
%pip install -q mediapipe==0.10.35 scikit-learn opencv-python-headless
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import csv, hashlib, json, shutil, urllib.request, warnings
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

SEED=42; SEQUENCE_FRAMES=17; MIN_DETECTED_FRAMES=12; FLOW_SIZE=128; FLOW_CLIP_PIXELS=20.0
BATCH_SIZE=8; EPOCHS=35; AUTOTUNE=tf.data.AUTOTUNE
CLASSES=["J","Z","other"]; CLASS_TO_ID={name:i for i,name in enumerate(CLASSES)}
tf.keras.utils.set_random_seed(SEED); np.random.seed(SEED)
DRIVE_ROOT=Path("/content/drive/MyDrive/proyecto-ia2")
VIDEO_ROOT=DRIVE_ROOT/"data/videos_jz"
WORK_DIR=DRIVE_ROOT/"data/processed/proyecto_def/video"
EXPORT_DIR=DRIVE_ROOT/"exports/proyecto_def/video_balanced_v2"
MANIFEST=WORK_DIR/"video_manifest.csv"; SPLIT_FILE=WORK_DIR/"split_balanced_v2.csv"
MODEL_ASSET=DRIVE_ROOT/"exports/proyecto_def/hand_landmarker.task"
TEMPORAL_MODEL=EXPORT_DIR/"jz_flow_cnn.keras"
WORK_DIR.mkdir(parents=True,exist_ok=True); EXPORT_DIR.mkdir(parents=True,exist_ok=True)
if not VIDEO_ROOT.is_dir(): raise FileNotFoundError(f"No encuentro {VIDEO_ROOT}")
if not MODEL_ASSET.exists():
    MODEL_ASSET.parent.mkdir(parents=True,exist_ok=True)
    url="https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"
    urllib.request.urlretrieve(url,MODEL_ASSET)
print("Videos:",VIDEO_ROOT,"\nSe esperan J/<signer>/clip, Z/<signer>/clip, other/<signer>/clip")
print("Los IDs de participante deben ser globales y repetirse en J y Z para la misma persona.")
print("Procesamiento y checkpoints:",EXPORT_DIR)


## Procesar videos a flujos ópticos

Lee `.mp4`, `.avi`, `.mov` y `.mkv`. Cada video se reduce a 17 fotogramas uniformes y produce 16 mapas de flujo de 128×128×2. Si los metadatos de fotogramas son incorrectos o no se puede buscar por índice, hace dos pasadas secuenciales para muestrear el clip. Se conserva un clip si MediaPipe detecta la mano en al menos 12 de los 17 fotogramas; los fotogramas con detección fallida no descartan por sí solos toda la secuencia.

In [ ]:
VIDEO_EXTENSIONS={".mp4",".avi",".mov",".mkv"}
def canonical_label(name):
    value=name.strip().lower()
    return {"j":"J","z":"Z","other":"other","none":"other","background":"other"}.get(value)

def sample_frames(path,count=SEQUENCE_FRAMES):
    cap=cv2.VideoCapture(str(path))
    if not cap.isOpened(): cap.release(); return None
    total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total>=2:
        indices=np.linspace(0,total-1,count).round().astype(int); frames=[]
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES,int(idx)); ok,frame=cap.read()
            if not ok: frames=[]; break
            frames.append(frame)
        cap.release()
        if len(frames)==count: return frames
    else:
        cap.release()

    # Fallback for playable files (often AVI) whose frame-count metadata is zero
    # or whose codec cannot seek reliably by frame index. Count with sequential
    # reads, reopen, then sample evenly without retaining the whole video in RAM.
    cap=cv2.VideoCapture(str(path))
    if not cap.isOpened(): cap.release(); return None
    actual_total=0
    while True:
        ok,_=cap.read()
        if not ok: break
        actual_total+=1
    cap.release()
    if actual_total<2: return None
    indices=np.linspace(0,actual_total-1,count).round().astype(int)
    wanted=set(indices.tolist()); selected={}
    cap=cv2.VideoCapture(str(path))
    if not cap.isOpened(): cap.release(); return None
    frame_index=0
    while True:
        ok,frame=cap.read()
        if not ok: break
        if frame_index in wanted: selected[frame_index]=frame.copy()
        frame_index+=1
    cap.release()
    if not wanted.issubset(selected): return None
    return [selected[int(idx)] for idx in indices]

def get_points(frame,detector):
    rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    result=detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb))
    if not result.hand_landmarks: return None
    h,w=frame.shape[:2]
    return np.asarray([(p.x*w,p.y*h) for p in result.hand_landmarks[0]],dtype=np.float32)

def fixed_crops(frames,points_by_frame):
    all_points=np.concatenate(points_by_frame,axis=0)
    x0,y0=all_points.min(axis=0); x1,y1=all_points.max(axis=0)
    cx,cy=(x0+x1)/2,(y0+y1)/2; side=max(float(max(x1-x0,y1-y0)*1.7),32.0)
    left,top=int(np.floor(cx-side/2)),int(np.floor(cy-side/2)); side=int(np.ceil(side))
    crops=[]
    for frame in frames:
        h,w=frame.shape[:2]; right,bottom=left+side,top+side
        pl,pt=max(0,-left),max(0,-top); pr,pb=max(0,right-w),max(0,bottom-h)
        padded=cv2.copyMakeBorder(frame,pt,pb,pl,pr,cv2.BORDER_CONSTANT,value=(0,0,0))
        crop=padded[top+pt:bottom+pt,left+pl:right+pl]
        crops.append(cv2.resize(crop,(FLOW_SIZE,FLOW_SIZE),interpolation=cv2.INTER_AREA))
    return crops

def video_to_flow(path,detector):
    frames=sample_frames(path)
    if frames is None: return None,"video ilegible o con menos de dos fotogramas"
    points=[get_points(frame,detector) for frame in frames]
    detected_points=[p for p in points if p is not None]
    if len(detected_points)<MIN_DETECTED_FRAMES:
        return None,f"MediaPipe detectó {len(detected_points)}/{SEQUENCE_FRAMES} fotogramas; mínimo {MIN_DETECTED_FRAMES}"
    # El recorte común usa todas las detecciones disponibles. Los fotogramas donde
    # falló MediaPipe se conservan para el flujo óptico; no invalidan el clip solos.
    crops=fixed_crops(frames,detected_points); gray=[cv2.cvtColor(c,cv2.COLOR_BGR2GRAY) for c in crops]
    fields=[]
    for previous,current in zip(gray[:-1],gray[1:]):
        flow=cv2.calcOpticalFlowFarneback(previous,current,None,0.5,3,15,3,5,1.2,0)
        fields.append(np.clip(flow/FLOW_CLIP_PIXELS,-1.0,1.0).astype(np.float32))
    return np.stack(fields),""

def prepare_video_flows():
    old={}
    if MANIFEST.exists():
        old_df=pd.read_csv(MANIFEST).drop_duplicates("video_rel",keep="last")
        old={str(r.video_rel):r._asdict() for r in old_df.itertuples(index=False)}
    video_files=sorted(p for p in VIDEO_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)
    if not video_files: raise ValueError("No hay videos compatibles en VIDEO_ROOT")
    header=not MANIFEST.exists() or MANIFEST.stat().st_size==0
    options=mp.tasks.vision.HandLandmarkerOptions(base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_ASSET)),
        running_mode=mp.tasks.vision.RunningMode.IMAGE,num_hands=1,min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5)
    detector=mp.tasks.vision.HandLandmarker.create_from_options(options); rows=[]
    try:
        with MANIFEST.open("a",newline="",encoding="utf-8") as stream:
            writer=csv.DictWriter(stream,fieldnames=["video_rel","label","signer","flow_rel","ok","error"])
            if header: writer.writeheader()
            for i,path in enumerate(video_files,1):
                rel=path.relative_to(VIDEO_ROOT); parts=rel.parts
                label=canonical_label(parts[0]) if len(parts)>=3 else None
                if label is None: warnings.warn(f"Ignoro ruta; debe ser clase/signante/video: {rel}"); continue
                signer=parts[1]
                if signer.lower() in {"unknown","desconocido","na"}:
                    warnings.warn(f"El ID de signante no es fiable: {rel}")
                out_name=hashlib.sha1(rel.as_posix().encode()).hexdigest()+".npy"
                out_rel=f"flows/{label}/{signer}/{out_name}"; output=WORK_DIR/out_rel
                prev=old.get(rel.as_posix())
                if prev and output.is_file() and int(prev["ok"])==1:
                    rows.append(prev); continue
                output.parent.mkdir(parents=True,exist_ok=True)
                flow,error=video_to_flow(path,detector); ok=flow is not None
                if ok: np.save(output,flow)
                record={"video_rel":rel.as_posix(),"label":label,"signer":signer,
                        "flow_rel":out_rel,"ok":int(ok),"error":error}
                writer.writerow(record); rows.append(record); stream.flush()
                if i%10==0 or i==len(video_files): print(f"{i}/{len(video_files)} | {rel} | {'OK' if ok else error}")
    finally: detector.close()
    return pd.DataFrame(rows).drop_duplicates("video_rel",keep="last")

manifest_df=prepare_video_flows()
print("Conteos procesados:")
display(manifest_df[manifest_df.ok.astype(int)==1].groupby(["label","signer"]).size().unstack(fill_value=0))


## Split por signante y entrenamiento

Se intenta construir un split por signante con las tres etiquetas presentes en train, validation y test. Un clip se conserva si MediaPipe detecta mano en al menos 12 de los 17 fotogramas; los fotogramas con detección fallida se mantienen en el cálculo de flujo. La CNN 3D usa pesos de clase calculados sobre train. Si el split no es posible, agrega variedad real por clase/persona; no asignes un ID inventado para simular signantes.

In [ ]:
usable=manifest_df[manifest_df.ok.astype(int)==1].copy()
if not set(CLASSES).issubset(set(usable.label)):
    raise ValueError(f"Se requieren clips procesables para {CLASSES}; conteos: {usable.label.value_counts().to_dict()}")
if usable.signer.nunique()<3:
    raise ValueError("Se requieren al menos tres IDs de signante; crea carpetas reales por persona.")

if SPLIT_FILE.exists():
    split_df=pd.read_csv(SPLIT_FILE)
    if set(split_df.video_rel)!=set(usable.video_rel):
        raise ValueError("Cambió el conjunto de videos desde el split guardado. Usa split_balanced_v2.csv nuevo o revisa la versión existente.")
else:
    # Asigna signantes completos y busca un reparto cercano a 70/15/15 por clase.
    # Validation y test deben tener al menos cinco clips de cada etiqueta.
    signer_names=sorted(usable.signer.unique())
    group_counts=np.zeros((len(signer_names),len(CLASSES)),dtype=np.int32)
    for gi,signer in enumerate(signer_names):
        counts=usable[usable.signer==signer].label.value_counts()
        for ci,label in enumerate(CLASSES): group_counts[gi,ci]=int(counts.get(label,0))
    fractions=np.asarray([0.70,0.15,0.15],dtype=np.float64)
    totals=group_counts.sum(axis=0).astype(np.float64)
    targets=fractions[:,None]*totals[None,:]
    rng=np.random.default_rng(SEED)
    best_assignment=None; best_score=np.inf
    for _ in range(100000):
        assignment=rng.choice(3,size=len(signer_names),p=fractions)
        counts=np.stack([group_counts[assignment==split].sum(axis=0) for split in range(3)])
        if np.any(counts[0]<1) or np.any(counts[1:]<5): continue
        relative=(counts-targets)/np.maximum(targets,1.0)
        score=float(np.square(relative).sum())
        if score<best_score:
            best_score=score; best_assignment=assignment.copy()
    if best_assignment is None:
        summary={name:{label:int(group_counts[i,j]) for j,label in enumerate(CLASSES) if group_counts[i,j]}
                 for i,name in enumerate(signer_names)}
        raise ValueError("No encontré split por signante con las tres clases y al menos 5 clips "
                         f"por clase en validación/prueba. Signantes y conteos: {summary}")
    split_names=np.asarray(["train","validation","test"])
    signer_to_split={name:split_names[best_assignment[i]] for i,name in enumerate(signer_names)}
    split_df=usable.copy(); split_df["split"]=split_df.signer.map(signer_to_split)
    split_df.to_csv(SPLIT_FILE,index=False)
    print("Error relativo del balance por clase:",round(best_score,4))
display(split_df.groupby(["split","label"]).size().unstack(fill_value=0))
display(split_df.groupby("split").signer.unique().rename("signantes"))

def stack_split(frame):
    X=np.stack([np.load(WORK_DIR/p) for p in frame.flow_rel])
    y=frame.label.map(CLASS_TO_ID).to_numpy(dtype=np.int32)
    return X,y
train_df=split_df[split_df.split=="train"]; val_df=split_df[split_df.split=="validation"]
test_df=split_df[split_df.split=="test"]
X_train,y_train=stack_split(train_df); X_val,y_val=stack_split(val_df); X_test,y_test=stack_split(test_df)
class_weights=compute_class_weight(class_weight="balanced",classes=np.arange(len(CLASSES)),y=y_train)
CLASS_WEIGHTS={i:float(w) for i,w in enumerate(class_weights)}
print("Class weights:",CLASS_WEIGHTS)

def build_model():
    inputs=tf.keras.Input((SEQUENCE_FRAMES-1,FLOW_SIZE,FLOW_SIZE,2),name="optical_flow_sequence")
    x=tf.keras.layers.Conv3D(16,(3,5,5),strides=(1,2,2),padding="same",activation="relu")(inputs)
    x=tf.keras.layers.MaxPool3D((1,2,2))(x)
    x=tf.keras.layers.Conv3D(32,(3,3,3),padding="same",activation="relu")(x)
    x=tf.keras.layers.MaxPool3D((2,2,2))(x)
    x=tf.keras.layers.Conv3D(64,(3,3,3),padding="same",activation="relu")(x)
    x=tf.keras.layers.GlobalAveragePooling3D()(x); x=tf.keras.layers.Dropout(0.35)(x)
    outputs=tf.keras.layers.Dense(3,activation="softmax",name="class_probs")(x)
    model=tf.keras.Model(inputs,outputs,name="jz_optical_flow_cnn")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return model

complete=EXPORT_DIR/"training_complete.json"
if TEMPORAL_MODEL.exists() and complete.exists():
    flow_model=tf.keras.models.load_model(TEMPORAL_MODEL,compile=False)
else:
    flow_model=build_model()
    ckpt_dir=EXPORT_DIR/"checkpoints"; ckpt_dir.mkdir(parents=True,exist_ok=True)
    callbacks=[
        tf.keras.callbacks.BackupAndRestore(backup_dir=str(EXPORT_DIR/"backup"),save_freq="epoch"),
        tf.keras.callbacks.ModelCheckpoint(str(ckpt_dir/"best.keras"),monitor="val_loss",mode="min",save_best_only=True,verbose=1),
        tf.keras.callbacks.ModelCheckpoint(str(ckpt_dir/"epoch-{epoch:02d}.weights.h5"),save_weights_only=True,save_best_only=False,save_freq="epoch"),
        tf.keras.callbacks.CSVLogger(str(EXPORT_DIR/"history.csv"),append=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=6,restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=2,min_lr=1e-6),
    ]
    flow_model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=EPOCHS,batch_size=BATCH_SIZE,
                   class_weight=CLASS_WEIGHTS,callbacks=callbacks,verbose=2)
    best=ckpt_dir/"best.keras"
    if best.exists(): flow_model=tf.keras.models.load_model(best,compile=False)
    flow_model.save(TEMPORAL_MODEL)
    (EXPORT_DIR/"classes.json").write_text(json.dumps(CLASSES,indent=2),encoding="utf-8")
    (EXPORT_DIR/"metadata.json").write_text(json.dumps({"classes":CLASSES,"sequence_frames":SEQUENCE_FRAMES,
        "flow_size":FLOW_SIZE,"flow_clip_pixels":FLOW_CLIP_PIXELS,"source":str(VIDEO_ROOT)},indent=2),encoding="utf-8")
    shutil.copy2(MODEL_ASSET,EXPORT_DIR/"hand_landmarker.task")
    complete.write_text(json.dumps({"complete":True}),encoding="utf-8")


## Evaluación final

Las métricas se calculan en signantes apartados. Debido al tamaño reducido y al origen heterogéneo de `other`, interpreta estos resultados como evaluación preliminar.

In [ ]:
probs=flow_model.predict(X_test,batch_size=BATCH_SIZE,verbose=0); pred=probs.argmax(axis=1)
metrics={"n_test":int(len(y_test)),"accuracy":float(accuracy_score(y_test,pred)),
         "macro_f1":float(f1_score(y_test,pred,average="macro",zero_division=0)),
         "classification_report":classification_report(y_test,pred,labels=[0,1,2],target_names=CLASSES,
             output_dict=True,zero_division=0)}
(EXPORT_DIR/"metrics.json").write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding="utf-8")
pd.DataFrame({"video_rel":test_df.video_rel,"signer":test_df.signer,"true":[CLASSES[i] for i in y_test],
              "prediction":[CLASSES[i] for i in pred]}).to_csv(EXPORT_DIR/"test_predictions.csv",index=False)
cm=confusion_matrix(y_test,pred,labels=[0,1,2],normalize="true")
fig,ax=plt.subplots(figsize=(6,5)); ConfusionMatrixDisplay(cm,display_labels=CLASSES).plot(ax=ax,cmap="Blues",colorbar=False,values_format=".2f")
fig.tight_layout(); fig.savefig(EXPORT_DIR/"confusion_matrix.png",dpi=150); plt.show()
print({k:v for k,v in metrics.items() if k!="classification_report"})
print("Modelo y checkpoints:",EXPORT_DIR)
